# PoliMillionaire baseline

Before running this notebook, put the files in Google Drive like this:

```
MyDrive/
└── Colab Notebooks/
    └── NLP_assignment/
        ├── poli_millionaire_clean_baseline_v2.ipynb
        └── millionaire_client/
            ├── __init__.py
            ├── client.py
            ├── auth.py
            ├── base.py
            ├── game.py
            ├── models.py
            ├── competitions.py
            ├── leaderboard.py
            └── exceptions.py
```

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import torch

Mounted at /content/gdrive/


In [3]:
BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('BASE_DIR not found. Create the NLP_assignment folder in Drive and upload the notebook there.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print('Path added successfully.')

BASE_DIR exists: True
BASE_DIR contents: ['PoliMillionaire.ipynb', '.DS_Store', 'millionaire_client', '.ipynb_checkpoints', 'test3_rag_game_runs', 'test3_offline_rag_index.pkl']
PACKAGE_DIR exists: True
PACKAGE_DIR contents: ['base.py', 'leaderboard.py', 'auth.py', 'competitions.py', 'client.py', 'game.py', '__init__.py', 'exceptions.py', 'models.py', '__pycache__']
Path added successfully.


In [4]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

In [6]:
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'gary'
PASSWORD = '13790229'

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

Logged in as: gary


In [26]:
competitions = client.competitions.list_all()
for c in competitions:
    print(c.id, c.name, c.max_levels)

COMPETITION_ID = competitions[2].id

0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15


In [8]:
model_id = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [9]:
def extract_letter(text):
    text = text.strip().upper()
    match = re.search(r'\b([ABCD])\b', text)
    if match:
        return match.group(1)
    if text and text[0] in 'ABCD':
        return text[0]
    return 'A'

def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, 'A', 'fallback'

    prompt = f'''PoliMillionaire MCQ. Reply only with A, B, C, or D.


Question: {question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Answer:'''

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    letter = extract_letter(text)
    idx = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[idx].id, letter, text

In [29]:
game = client.game.start(competition_id=COMPETITION_ID, mode='text')

while game.in_progress:
    question = game.current_question
    if question is None:
        break

    print('Level:', game.current_level)
    print(question.text)
    for i, opt in enumerate(question.options):
        print(f"{chr(65+i)}) {opt.text}")

    option_id, letter, raw = choose_answer(question)
    print('Predicted:', letter, '| Raw output:', raw)

    try:
        result = game.answer(option_id)
    except TimeoutError:
        print('Timed out')
        break
    except RateLimitError:
        print('Rate limited, waiting...')
        time.sleep(5)
        result = game.answer(option_id)

    print('Correct:', result.correct, '| Earned:', result.earned_amount)

    if result.game_over:
        break

    time.sleep(0.5)

print('Final earned:', game.earned_amount)

Level: 1
The populations of black rhinoceroses have been impacted by human interactions to the point of being endangered. Which interaction best explains why the black rhino is now endangered?
A) over hunting
B) water pollution
C) airborne diseases
D) increased deforestation
Predicted: A | Raw output: A
Correct: True | Earned: 100
Level: 2
Which list describes the particles that make up an atom?
A) protons, neutron, and electrons
B) nucleus, electron cloud, and energy levels
C) ionic bonds, covalent bonds, and ions
D) nucleus, electrons, and quarks
Predicted: A | Raw output: A
Correct: True | Earned: 200
Level: 3
What type of film is Sausage Party?
A) Animated documentary
B) Animated drama
C) Animated thriller
D) Animated comedy
Predicted: D | Raw output: D
Correct: True | Earned: 300
Level: 4
In the water cycle, after water has condensed to form clouds, it falls back to Earth in the form of
A) condensation.
B) precipitation.
C) vaporization.
D) evaporation.
Predicted: B | Raw output: 